In [1]:
import numpy as np
import pandas as pd

input_file = "train.csv"
df_train = pd.read_csv(input_file)
df_train.shape

(7352, 563)

In [2]:
df_train["subject"].value_counts()

subject
25    409
21    408
26    392
30    383
28    382
27    376
23    372
17    368
16    366
19    360
1     347
29    344
3     341
15    328
6     325
14    323
22    321
11    316
7     308
5     302
8     281
Name: count, dtype: int64

In [3]:
df_train["Activity"].value_counts()

Activity
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64

In [4]:
from sklearn.preprocessing import OrdinalEncoder

df_train_cat = df_train[["Activity"]]
ordinal_encoder = OrdinalEncoder()
y_train = ordinal_encoder.fit_transform(df_train_cat).flatten()
ordinal_encoder.categories_

[array(['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS',
        'WALKING_UPSTAIRS'], dtype=object)]

In [5]:
# 특성 엔지니어링: 각 행(측정 구간) 하나만 보고 만드는 특성 (다른 행·전체 데이터 통계를 쓰지 않으므로 누수 없음)
#  원본 561 특성에 부호를 유지한 제곱근 변환 값을 덧붙인다
def har_features(X):
    return np.hstack([X, np.sign(X) * np.sqrt(np.abs(X))])

X_raw = df_train.drop(["subject", "Activity"], axis=1).to_numpy()
har_features(X_raw).shape

(7352, 1122)

In [6]:
# 1) 모델 비교: 원본 특성 vs 내 특성, 여러 분류 모델 (10-fold, shuffle=True)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

kf = KFold(n_splits=10, shuffle=True, random_state=42)

models = {
    "SGD": SGDClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=3000),
    "LinearSVC": LinearSVC(max_iter=10000),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "SVC": SVC(C=10),
    "LGBM": LGBMClassifier(n_estimators=600, learning_rate=0.05, colsample_bytree=0.3, verbose=-1, n_jobs=1),
}
for name, model in models.items():
    raw = cross_val_score(model, X_raw, y_train, cv=kf, scoring="accuracy", n_jobs=-1).mean()
    fe = cross_val_score(make_pipeline(FunctionTransformer(har_features), model), X_raw, y_train, cv=kf, scoring="accuracy", n_jobs=-1).mean()
    print(f"{name:20s} 원본 {raw:.4f}   내 특성 {fe:.4f}")

SGD                  원본 0.9771   내 특성 0.9754


LogisticRegression   원본 0.9848   내 특성 0.9842


LinearSVC            원본 0.9853   내 특성 0.9841


DecisionTree         원본 0.9441   내 특성 0.9431


RandomForest         원본 0.9822   내 특성 0.9812


SVC                  원본 0.9876   내 특성 0.9857


/usr/local/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


LGBM                 원본 0.9948   내 특성 0.9944


In [7]:
# 2) 하이퍼파라미터 탐색: 가장 좋은 SVC 의 C, gamma (Grid Search, 10-fold shuffle=True)
#  SVC 는 원본 특성에서 더 좋았으므로 원본 특성으로 탐색
from sklearn.model_selection import GridSearchCV

param_grid = {"C": [10, 30, 100], "gamma": ["scale", 0.01, 0.02, 0.03]}
grid_search = GridSearchCV(SVC(), param_grid, cv=kf, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_raw, y_train)
print(grid_search.best_params_, grid_search.best_score_)
for mean, params in zip(grid_search.cv_results_["mean_test_score"], grid_search.cv_results_["params"]):
    print(f"{mean:.4f}", params)

{'C': 30, 'gamma': 0.02} 0.9930628881987577
0.9876 {'C': 10, 'gamma': 'scale'}
0.9898 {'C': 10, 'gamma': 0.01}
0.9912 {'C': 10, 'gamma': 0.02}
0.9925 {'C': 10, 'gamma': 0.03}
0.9895 {'C': 30, 'gamma': 'scale'}
0.9910 {'C': 30, 'gamma': 0.01}
0.9931 {'C': 30, 'gamma': 0.02}
0.9927 {'C': 30, 'gamma': 0.03}
0.9903 {'C': 100, 'gamma': 'scale'}
0.9914 {'C': 100, 'gamma': 0.01}
0.9927 {'C': 100, 'gamma': 0.02}
0.9927 {'C': 100, 'gamma': 0.03}


In [8]:
# 3) 최종 모델: 튜닝한 SVC + LGBM 을 Stacking (최종 판정기 LogisticRegression), 10-fold 교차 검증 (shuffle=True)
from sklearn.ensemble import StackingClassifier

kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
final_clf = StackingClassifier(
    estimators=[("svc", grid_search.best_estimator_),
                ("lgbm", LGBMClassifier(n_estimators=1500, learning_rate=0.05, colsample_bytree=0.3, verbose=-1, n_jobs=1))],
    final_estimator=LogisticRegression(C=1, max_iter=5000),
    cv=kf5)

#  cross_val_predict 로 fold 별 예측을 한 번에 구하고, 같은 fold 로 정확도와 오차 행렬을 계산
from sklearn.model_selection import cross_val_predict

y_pred = cross_val_predict(final_clf, X_raw, y_train, cv=kf, n_jobs=-1)
cv_res = np.array([(y_pred[te] == y_train[te]).mean() for _, te in kf.split(X_raw)])
cv_res

array([0.99728261, 0.99728261, 0.99183673, 0.9877551 , 0.99455782,
       0.99727891, 0.99591837, 0.99863946, 1.        , 0.99727891])

In [9]:
np.mean(cv_res)

np.float64(0.9957830523513754)

In [10]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_train, y_pred)

array([[1407,    0,    0,    0,    0,    0],
       [   0, 1271,   14,    0,    0,    1],
       [   0,   16, 1358,    0,    0,    0],
       [   0,    0,    0, 1226,    0,    0],
       [   0,    0,    0,    0,  986,    0],
       [   0,    0,    0,    0,    0, 1073]])